In [39]:
import tensorflow as tf
from tensorflow.keras import mixed_precision 
import keras

gpus = tf.config.list_physical_devices("GPU")
tf.config.set_logical_device_configuration(
    gpus[0],
    [tf.config.LogicalDeviceConfiguration(memory_limit=5120)]
)
                                         
policy = mixed_precision.Policy('mixed_float16')  
mixed_precision.set_global_policy(policy)  

In [73]:
from datasets import load_dataset
import numpy as np
dataset = open("input.txt",encoding="utf-8").read()

chars = sorted(set(dataset))
vocab_size = len(chars)

vectorizer = {ch:i for i,ch in enumerate(chars)}
devectorizer = {i:ch for i,ch in enumerate(chars)}

encode = lambda x: [vectorizer[a] for a in x]
decode = lambda y: [devectorizer[b] for b in y]

split = int(0.9*len(text))

data = np.array(encode(dataset))

In [90]:
BLOCK_SIZE = 64
BATCH_SIZE = 256
EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 512        
NUM_LAYERS = 4
EPOCHS = 15

In [91]:
def mk_data(data, block_size, batch_size):
    data = tf.reshape(data, [-1])

    ds = tf.data.Dataset.from_tensor_slices(data)

    ds = ds.window(block_size + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda x: x.batch(block_size + 1))

    ds = ds.map(lambda seq: (seq[:-1], seq[1:]))

    ds = ds.cache()
    
    ds = ds.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds


train_ds = mk_data(data[:split], BLOCK_SIZE, BATCH_SIZE)
val_ds = mk_data(data[split:], BLOCK_SIZE, BATCH_SIZE)

In [92]:
from tensorflow.keras import layers

class CausalSelfAttention(layers.Layer):
    def __init__(self, embed_dim, block_size, num_heads, dropout=0.3):
        super().__init__()
        self.mha = layers.MultiHeadAttention(
            num_heads = num_heads,
            key_dim = embed_dim // num_heads,
            dropout = dropout
        )
        self.layernorm = layers.LayerNormalization()
        self.dropout = layers.Dropout(dropout)

    def call(self, x):
        attn_op = self.mha(query=x, key=x, value=x, use_causal_mask=True)
        return self.layernorm(x + self.dropout(attn_op))

In [93]:
class FeedForwardLayer(layers.Layer):
    def __init__(self, ff_dim, embed_dim, dropout=0.3):
        super().__init__()
        self.network = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dropout(dropout),  # Increased from 0.2
            layers.Dense(embed_dim),
            layers.Dropout(dropout)   # Increased from 0.2
        ])
        self.layernorm = layers.LayerNormalization()

    def call(self, x):
        return (self.layernorm(x + self.network(x)))

In [94]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, block_size):
        super().__init__()
        self.attnt = CausalSelfAttention(embed_dim,block_size,num_heads)
        self.ffn = FeedForwardLayer(ff_dim,embed_dim)
    
    def call(self,x):
        x = self.attnt(x)
        x = self.ffn(x)
        return x

In [95]:
class MiniGPT(tf.keras.Model):
    def __init__(self,embed_dim,num_heads,ff_dim,block_size,vocab_size,num_layers):
        super().__init__()
        self.token_emb = layers.Embedding(vocab_size,embed_dim)
        self.pos_emb = layers.Embedding(block_size,embed_dim)
        self.block = [TransformerBlock(embed_dim,num_heads,ff_dim,block_size) for _ in range(num_layers)]
        self.layernorm = layers.LayerNormalization()
        self.head = layers.Dense(vocab_size)
        
    def call(self,x):
        seq_len = tf.shape(x)[1]
        tok_emb = self.token_emb(x)

        pos = tf.range(seq_len)
        pos_emb = self.pos_emb(pos)

        x = tok_emb+pos_emb

        for block in self.block:
            x = block(x)
        
        x = self.layernorm(x)
        logits = self.head(x)

        return logits

In [96]:
model = MiniGPT(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    block_size=BLOCK_SIZE,
    num_layers=NUM_LAYERS
)

model.compile(
    loss = keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
        
    ),
    optimizer = tf.optimizers.Adam(
        learning_rate=1e-4
    ),
    metrics = ["accuracy"],
    jit_compile = True
)

model.summary()

Model: "mini_gpt_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_12 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_13 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_24            │ ?                      │   0 (unbuilt) │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_25            │ ?                      │   0 (unbuilt) │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_26            │ ?                      │   0 (unbuilt) │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_27            │ ?                      │   0 (unbuilt) │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_62          │ ?                      │   0 (unbuilt) │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_62 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [97]:
steps_per_epoch = (split-BLOCK_SIZE) // BATCH_SIZE
model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS,steps_per_epoch=steps_per_epoch)

Epoch 1/15


2026-03-16 16:24:43.099606: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-03-16 16:24:50.726085: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert


 4356/38424 ━━━━━━━━━━━━━━━━━━━━ 16:38 29ms/step - accuracy: 0.4032 - loss: 2.0626

2026-03-16 16:27:05.477842: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert


 4357/38424 ━━━━━━━━━━━━━━━━━━━━ 17:46 31ms/step - accuracy: 0.4032 - loss: 2.0625

...

2026-03-16 16:45:15.807377: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 15800771527829625575
2026-03-16 16:45:15.807419: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 930716250595523579


38424/38424 ━━━━━━━━━━━━━━━━━━━━ 91s 2ms/step - accuracy: 0.6568 - loss: 1.0940   
Epoch 15/15


2026-03-16 16:46:46.668925: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 15800771527829625575
2026-03-16 16:46:46.668967: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 930716250595523579


38424/38424 ━━━━━━━━━━━━━━━━━━━━ 90s 2ms/step - accuracy: 0.6604 - loss: 1.0828   


2026-03-16 16:48:16.541239: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 15800771527829625575
2026-03-16 16:48:16.541290: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 930716250595523579


In [100]:
def generate(prompt, length=500, temperature=0.8):
    ids = encode(prompt)
    for _ in range(length):
        x = tf.constant([ids[-BLOCK_SIZE:]])
        logits = model(x, training=False)[0, -1] / temperature
        ids.append(int(tf.random.categorical([logits], 1)[0, 0]))
    return "".join(decode(ids))

print(generate("ROMEO:\n"), end=" ")

ROMEO:
You have purposed,
Hark, widow Dido's son Lucentio, sparition
They are obey. They are not folks
They heavy easy means for your came at thy clouds!

ANTONIO:
Tunis as we'll be well enter--

SEBASTIAN:
What is indeast thou hear'st so; he will live
I would not infection of him: if now I loved you will so?

ANTONIO:
He doth I see of him.

ANTONIO:
It is not so?

FERDINAND:
I say, as you do him.

ANTONIO:
Is not language! Come and know
What is it the fearfully.

SEBASTIAN:
We are the five order-seeiz 